# 11 — Observabilité et diagnostic des performances

Analyse les traces produites par le pipeline : latence, cache, erreurs, routes, conversation et recherche exhaustive.

> Vérifiez toujours les drapeaux `RUN_*` avant l'exécution. Pour cette reconstruction propre, les traitements locaux nécessaires des étapes 02 à 07 sont déjà activés ; les collectes réseau et évaluations GPU facultatives restent désactivées.


In [ ]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.yaml').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Impossible de trouver config.yaml et src/.')

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Projet :', PROJECT_ROOT)
print('Python :', sys.executable)
print('Données actives :', PROJECT_ROOT / 'data')


In [ ]:
import pandas as pd
from config import load_config
from observability import load_traces, summarize_traces

config = load_config(PROJECT_ROOT / 'config.yaml')
events = load_traces(config.observability.traces_full_path)
display(pd.Series(summarize_traces(events)).to_frame('valeur'))
traces = pd.json_normalize(events, sep='.') if events else pd.DataFrame()
print('Événements :', len(traces))


In [ ]:
if not traces.empty:
    wanted = [column for column in [
        'timestamp_utc', 'question', 'status', 'metadata.route',
        'metadata.contextualized', 'metadata.expanded_retrieval',
        'retrieval_stats.total_seconds', 'generation_stats.total_seconds',
        'generation_stats.tokens_per_second', 'generation_stats.cache_hit',
        'citation_count', 'total_seconds', 'error',
    ] if column in traces]
    display(traces[wanted].tail(20))


In [ ]:
if not traces.empty and 'total_seconds' in traces:
    display(traces.nlargest(10, 'total_seconds')[wanted])
    if 'metadata.route' in traces:
        display(traces.groupby('metadata.route')['total_seconds'].agg(['count', 'mean', 'median', 'max']))


## Lecture recommandée

Utilisez P50 pour l'expérience habituelle et P95 pour les lenteurs rares. Séparez retrieval et génération avant d'optimiser. Une vitesse de génération anormalement basse peut signaler une contention VRAM ou un déport partiel en RAM.
